# Ejemplo: 32 MOC3D Problem 1

**Resumen**. 

Se resuelve un problema de flujo y transporte en una dimensión usando los datos del ejemplo descrito en [1]. Los resutlados se comparan con soluciones analíticas, las cuales están previamente calculas y se encuentran en la carpeta `analytic`.

<p xmlns:cc="http://creativecommons.org/ns#" xmlns:dct="http://purl.org/dc/terms/"><a property="dct:title" rel="cc:attributionURL" href="https://github.com/luiggix/mf6_tutorial/">MODFLOW 6: tutorial</a> (2_flow_trans/01_flow_trans_1D_exchange.ipynb) by <b>Luis M. de la Cruz Salas (2025)</b> is licensed under <a href="http://creativecommons.org/licenses/by-sa/4.0/?ref=chooser-v1" target="_blank" rel="license noopener noreferrer" style="display:inline-block;">Attribution-ShareAlike 4.0 International<img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/cc.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/by.svg?ref=chooser-v1"><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/sa.svg?ref=chooser-v1"></a>.</p> 

# Descripción.

<img src="figures/modflow_models_CD.png" width="500px" hspace="5" vspace="5" style="float: right; margin-left: 30px"/>
En este ejemplo se resolverá el flujo y transporte en 1D que están conectados mediante un intercambio (*exchange*) como se muestra en la figura D (Tomada de [2]). En este caso solo se define una sola simulación y a ella se le agregan las componentes para el flujo, modelo GWF, para el transporte, modelo GWT, y para el intercambio. 

**Obsérvese que cada modelos numérico tiene su propio modelo de solución**.


## Discretización del tiempo.

* La simulación consta de un período de estrés que tiene una duración de $1$ $day$.
* El período de estrés se divide en $240$ pasos de tiempo del mismo tamaño.

|Parameter | Value| Units | Variable |
|---:|:---:|:---:|:---|
|Number of stress periods (NPER) | $1$ | | `tdis["nper"]` |
|Total time | $1$ | day | `tdis["perioddata"][0][0]` |
|Number of time steps (NSTP) | $1$ | | `tdis["perioddata"][0][1]` |
|Multiplier (TSMULT)| $1$ | | `tdis["perioddata"][0][2]` |

## Discretización espacial.

* El modelo de malla consiste de $1$ capa, $30$ columnas y $1$ renglón.
* La longitud del renglón es de $30$ [m].
* La longitud de la columna es $1$ [m].
* Con la información anterior se calcula el ancho del renglón, `DELC`, y de las columnas, `DELR`, que ambos casos debe ser $1$ [m].
* La parte superior (`TOP`) de la celda es $1.0$ [m] y la parte inferior (`BOTTOM`) es cero.
* La longitud de la capa es igual a $1.0$ [m], valor que se calcula de `|TOP - BOTTOM|`.

|Parameter | Value| Units | Variable |
|---:|:---:|:---:|:---|
|Length of system (rows) | $30.0$ | m | |
|Number of layers | $1$ | | `dis["nlay"]`|
|Number of rows | $1$ | | `dis["nrow"]`|
|Number of columns | $30$ | | `dis["ncol"]`|
|Column width | $1$ | m | `dis["delr"]`|
|Row width | $1$ | m | `dis["delc"]`|
|Top of the model | $1$ | m | `dis["top"]`|
|Layer bottom elevation (m) | $0$ | m | `dis["botm"]`|

## Parámetros físicos.

* Al utilizar un valor de porosidad uniforme de $0.1$, se obtiene un valor de velocidad de $0.02$ $m/s$ como resultado de la inyección de agua a una velocidad de $0.02$ $m^3/s$ en la primera celda.
* A la última celda se le asigna una carga constante con un valor de $1.0 m$, aunque este valor no es importante ya que las celdas están marcadas como confinadas.
* A la concentración del agua inyectada se le asigna un valor de $1.0$, y cualquier agua que salga a través de la celda de carga constante sale con la concentración simulada del agua en esa última celda.
* La advección se resuelve utilizando el esquema **TVD** para reducir la dispersión numérica.

|Parameter | Value| Units | Variable |
|---:|:---:|:---:|:---|
|Specific discharge |0.2| m/d |`phys["specific_discharge"]`| 
|Hydraulic conductivity |1.0| m/d | `phys["hydraulic_conductivity"]` |
|Source concentration |1.0| unitless | `phys["source_concentration"]` |
|Porosity | 0.5 | unitless |  `phys["porosity"]` |
|Initial Concentration | 0.0 | unitless | `phys["initial_concentration"]` |
|Longitudinal Dispersivity | 0.1 |  | `phys["longitudinal_dispersivity"]` |
|Retardation Factor | 1.0 |  | `phys["'retardation_factor"]` |
|Decay Rate | 0.0 |  | `phys["decay_rate"]` |
|Dispersion coefficient | | | `phys["dispersion_coefficient"]` |

$$
\text{Dispersion Coefficient} = \dfrac{\text{Longitudinal Dispersivity} \times \text{Specific Discharge}}{\text{Retardation Factor}}
$$

### Cuatro casos:
<img src="figures/moc3d.png" width=600px>

# Implementación

## Paso 0. Importación de bibliotecas

In [25]:
import os
import numpy as np
import matplotlib.pyplot as plt
import flopy
import xmf6

## Paso 1. Parámetros físicos.

In [28]:
phys = dict(
    specific_discharge = 0.2,  # Specific discharge ($m/d$)
    hydraulic_conductivity = 1.0,  # Hydraulic conductivity ($m/d$)
    source_concentration = 1.0,  # Source concentration (unitless)
    porosity = 0.5,  # Porosity of mobile domain (unitless)
    initial_concentration = 0.0,  # Initial concentration (unitless)
)
xmf6.nice_print(phys, "Parámetros físicos")


Parámetros físicos
――――――――――――――――――
    specific_discharge = 0.2
hydraulic_conductivity = 1.0
  source_concentration = 1.0
              porosity = 0.5
 initial_concentration = 0.0
――――――――――――――――――


## Paso 2. Datos para la simulación del flujo: GWF

In [31]:
# Parámetros de la simulación (flopy.mf6.MFSimulation)
init_f = {
    'sim_name' : "flow",
    'exe_name': r"mf6/bin/mf6.exe",
    'sim_ws' : "output_ft_ex"
}

# Parámetros para el tiempo (flopy.mf6.ModflowTdis)
tdis_f = {
    'units': "days",
    'nper' : 1,
    'perioddata': [(1.0, 1, 1.0)] #(1.0,duración del periodo,1,num.de.pasos.de.tiempo,1.0,multiplicador)
}

# Parámetros para la solución numérica (flopy.mf6.ModflowIms)
# OJO: definiremos el modelo de solución más adelante
#ims = {}

# Parámetros para el modelo de flujo (flopy.mf6.ModflowGwf)
gwf = { 
    'modelname': init_f["sim_name"],
    'save_flows': True
}

# Parámetros para la discretización espacial (flopy.mf6.ModflowGwfdis)
dis = {
    'length_units' : "meters", # "centimeters",
    'nlay': 1, 
    'nrow': 1, 
    'ncol': 30, #120,
    'delr': 1.0, # 0.1, 
    'delc': 1.0, #0.1, 
    'top' : 1.0, 
    'botm': 0.0 
}

# Parámetros para las condiciones iniciales (flopy.mf6.ModflowGwfic)
ic = {
    'strt': 1.0
}

# Parámetros para las condiciones de frontera (flopy.mf6.ModflowGwfchd)
chd = {
    'stress_period_data': [[(0, 0, dis['ncol'] - 1), 1.0]],     
}

# Parámetros para las propiedades de flujo (flopy.mf6.ModflowGwfnpf)
npf = {
    'save_specific_discharge': True,
    'save_saturation': True,
    'icelltype':  0,
    'k': phys["hydraulic_conductivity"] # Hydraulic conductivity ($m s^{-1}$) 
}

# Parámetros para las propiedades de los pozos (flopy.mf6.ModflowGwfwel)
q = phys["specific_discharge"] * dis['delc'] * dis['delr'] * dis['top']
well = {
    'stress_period_data': [[(0, 0, 0), q, phys["source_concentration"],]],
    'pname': "WEL-1",
    'auxiliary' : ["CONCENTRATION"],
#    'save_flows': True
}

# Parámetros para almacenar y mostrar la salida de la simulación (flopy.mf6.ModflowGwfoc)
oc = {
    'budget_filerecord': f"{init_f['sim_name']}.bud",
    'head_filerecord': f"{init_f['sim_name']}.hds",
    'saverecord': [("HEAD", "ALL"), ("BUDGET", "ALL")],
#    'printrecord': [("HEAD", "ALL")]
}

# --- Inicialización de la simulación ---
o_sim = xmf6.common.init_sim(silent = True, init = init_f, tdis = tdis_f)
o_gwf, packages = xmf6.gwf.set_packages(o_sim, silent = True,
                                        gwf = gwf, 
                                        dis = dis, ic = ic, chd = chd, npf = npf, oc = oc, well = well)

# Definición del modelo de solución. Se realiza en este punto porque primero
# se requiere definir el objeto de flujo 'o_gwf' para conocer el nombre y
# posteriormente hacer el "registro" del modelo de solución en el objeto 'o_sim'.
o_ims_f = flopy.mf6.ModflowIms(
    o_sim,
    print_option="ALL",
    under_relaxation="NONE",
    linear_acceleration="BICGSTAB",
    scaling_method="NONE",
    reordering_method="NONE",
    filename=f"{o_gwf.name}.ims",
)
o_sim.register_ims_package(o_ims_f, [o_gwf.name])


# Referencias

[1] MODFLOW 6 – Example problems, MODFLOW 6 Development Team, with contributions from Chieh Ying Chen and Mike Toews 02/07/2024.**  **32 MOC3D Problem 1**. (Archivo: `mf6examples.pdf` del directorio `doc` de la distribución de MODFLOW 6).

[2] Langevin, C. D., Hughes, J. D., Provost, A. M., Russcher, M. J., & Panday, S. (2023). MODFLOW as a configurable Multi‐Model Hydrologic Simulator. Ground Water. https://doi.org/10.1111/gwat.13351.
